# CAE Colab Run

This notebook runs the CAE (Convolutional Autoencoder) SSL pipeline: pretraining → fine-tuning → evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime

REPO_URL = 'https://github.com/satvikkaul/SSL_Prostate_Cancer_Grading.git'
BRANCH = 'method/moco-v2'
PROJECT_DIR = '/content/SSL_Prostate_Cancer_Grading'
DRIVE_ROOT = '/content/drive/MyDrive/Prostate_SSL'
DATASET_ZIP = f'{DRIVE_ROOT}/dataset.zip'
RUN_NAME = f'cae_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
DRIVE_RUN_DIR = f'{DRIVE_ROOT}/runs/{RUN_NAME}'
os.environ['MPLCONFIGDIR'] = '/tmp/mplconfig'

print('PROJECT_DIR =', PROJECT_DIR)
print('DATASET_ZIP =', DATASET_ZIP)
print('DRIVE_RUN_DIR =', DRIVE_RUN_DIR)

In [ ]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone -b "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!git branch --show-current
!git log -1 --oneline

## Install Dependencies

In [ ]:
%cd "$PROJECT_DIR"
!pip install -q openpyxl scikit-learn pandas matplotlib pillow opencv-python

## Setup Dataset

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./dataset'):
    shutil.rmtree('./dataset')

!unzip -q "$DATASET_ZIP" -d .
!python data/setup.py

## Link Output to Drive

In [ ]:
%cd "$PROJECT_DIR"
import shutil

if os.path.exists('./output'):
    shutil.rmtree('./output')

os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
os.symlink(DRIVE_RUN_DIR, './output')
print(f'Linked ./output -> {DRIVE_RUN_DIR}')

## Apply Patches for CAE

Add CLI arguments to train_cae.py and fix finetune_cae.py to auto-detect latest weights.

In [ ]:
%cd "$PROJECT_DIR"
import re
import glob

# Patch train_cae.py to add CLI arguments
train_cae_path = 'training/cae/train_cae.py'
with open(train_cae_path, 'r') as f:
    content = f.read()

if 'argparse' not in content:
    # Add argparse import
    content = content.replace(
        'import sys',
        'import sys\nimport argparse'
    )
    
    # Add argument parser right after the print_device_configuration function
    parser_code = '''

def parse_args():
    parser = argparse.ArgumentParser(description="CAE SSL pretraining")
    parser.add_argument('--epochs', type=int, default=15, help='Number of training epochs')
    parser.add_argument('--batch_size', type=int, default=16, help='Batch size')
    parser.add_argument('--lr', type=float, default=0.0005, help='Learning rate')
    return parser.parse_args()

args = parse_args()
'''
    # Insert right before the hyperparameters comment
    content = re.sub(
        r'(#%%\n# Hyper-parametrs)',
        parser_code + r'\1',
        content
    )
    
    # Replace hardcoded values
    content = content.replace('batch_size = 16', 'batch_size = args.batch_size')
    content = content.replace('epochs = 15  # Reduced from 5', 'epochs = args.epochs  # Reduced from 5')
    content = content.replace('lr = 0.0005', 'lr = args.lr')
    
    with open(train_cae_path, 'w') as f:
        f.write(content)
    print('✓ Patched train_cae.py with CLI arguments')
else:
    print('✓ train_cae.py already patched')

# Patch finetune_cae.py to auto-detect latest weights and add CLI args
finetune_cae_path = 'training/cae/finetune_cae.py'
with open(finetune_cae_path, 'r') as f:
    content = f.read()

if 'argparse' not in content:
    # Add argparse import
    content = content.replace(
        'import sys',
        'import sys\nimport argparse'
    )
    
    # Add argument parser
    parser_code = '''

def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tune CAE encoder")
    parser.add_argument('--epochs_stage1', type=int, default=50, help='Stage 1 epochs (frozen encoder)')
    parser.add_argument('--epochs_stage2', type=int, default=0, help='Stage 2 epochs (unfrozen encoder)')
    parser.add_argument('--batch_size', type=int, default=8, help='Batch size')
    parser.add_argument('--weights', type=str, default='auto', help='Path to CAE weights (auto=latest)')
    return parser.parse_args()

args = parse_args()
'''
    # Insert before CONFIGURATION
    content = re.sub(
        r'(# --- CONFIGURATION ---)',
        parser_code + r'\1',
        content
    )
    
    # Replace hardcoded values
    content = content.replace('BATCH_SIZE = 8', 'BATCH_SIZE = args.batch_size')
    content = content.replace('EPOCHS_STAGE_1 = 50', 'EPOCHS_STAGE_1 = args.epochs_stage1')
    content = content.replace('EPOCHS_STAGE_2 = 0', 'EPOCHS_STAGE_2 = args.epochs_stage2')
    
    # Replace hardcoded weights path with auto-detection
    old_weights = "WEIGHTS_PATH = './output/models/exp_0012/weights/VAE.weights.h5'  # Updated to new trained model"
    new_weights = '''# Auto-detect latest CAE weights if not specified
if args.weights == 'auto':
    weight_candidates = sorted(glob.glob('./output/models/exp_*/weights/VAE.weights.h5'))
    if not weight_candidates:
        print("ERROR: No CAE weights found. Run train_cae.py first!")
        exit()
    WEIGHTS_PATH = weight_candidates[-1]
    print(f"Auto-detected weights: {WEIGHTS_PATH}")
else:
    WEIGHTS_PATH = args.weights'''
    
    content = content.replace(old_weights, new_weights)
    
    # Add glob import at top if not present
    if 'import glob' not in content:
        content = content.replace('import os', 'import os\nimport glob')
    
    with open(finetune_cae_path, 'w') as f:
        f.write(content)
    print('✓ Patched finetune_cae.py with CLI arguments and auto-detection')
else:
    print('✓ finetune_cae.py already patched')

## Pretrain CAE

Train the autoencoder for image reconstruction (SSL pretraining).

In [ ]:
%cd "$PROJECT_DIR"
EPOCHS = 30         # Increase from default 15
BATCH_SIZE = 32     # T4 can handle 32
LR = 0.0005         # Default learning rate

cmd = f'python training/cae/train_cae.py --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR}'
print(cmd)
!$cmd

## Fine-Tune CAE

Load pretrained encoder and train classification head. Captures both frozen-encoder (Stage 1) and optional end-to-end fine-tuning (Stage 2).

In [ ]:
%cd "$PROJECT_DIR"

FINETUNE_BATCH_SIZE = 32    # T4 can handle 32
EPOCHS_STAGE_1 = 50         # Head-only training
EPOCHS_STAGE_2 = 20         # End-to-end fine-tuning (set to 0 to skip)
WEIGHTS = 'auto'            # Auto-detect latest weights

cmd = (
    f'python training/cae/finetune_cae.py'
    f' --epochs_stage1 {EPOCHS_STAGE_1}'
    f' --epochs_stage2 {EPOCHS_STAGE_2}'
    f' --batch_size {FINETUNE_BATCH_SIZE}'
    f' --weights {WEIGHTS}'
)
print(cmd)
!$cmd

## Evaluate CAE

Generate evaluation metrics on the test set.

In [ ]:
%cd "$PROJECT_DIR"
!python evaluation/cae/eval_cae.py

## Sync Runtime State to Drive

In [ ]:
%cd "$PROJECT_DIR"

import shutil
from pathlib import Path

sync_root = Path(DRIVE_RUN_DIR) / "runtime_sync"
dataset_sync = sync_root / "dataset"
notebook_sync = sync_root / "notebook"

dataset_sync.mkdir(parents=True, exist_ok=True)
notebook_sync.mkdir(parents=True, exist_ok=True)

# Copy notebook if it exists in project dir (optional)
if Path("run_cae_colab.ipynb").exists():
    shutil.copy2("run_cae_colab.ipynb", notebook_sync / "run_cae_colab.ipynb")

for name in ["Train.csv", "Test.csv", "TrainSplit.csv", "Val.csv"]:
    src = Path("dataset") / name
    if src.exists():
        shutil.copy2(src, dataset_sync / name)

os.system(f'git rev-parse HEAD > "{sync_root / "commit.txt"}"')
os.system(f'find ./output -maxdepth 4 -type f | sort > "{sync_root / "output_manifest.txt"}"')

print(f"Synced runtime artifacts to: {sync_root}")

In [ ]:
%cd "$PROJECT_DIR"
!echo "Run directory: $DRIVE_RUN_DIR"
!find ./output -maxdepth 3 -type f | sort | tail -n 30